# Delving into some of the basic architectures of ANN and hyperparameter tunning with hosing pricing data.
The main goal of this notebook is to explore some of the most basic but common architectures, in term of neural networks, for regression tasks. In addition, it covers a first exploration of hyperparameter tuning, setting callbacks, and structuting the models from rigid to flexible frameworks within ketas.

In [1]:
# -- Basic libraries --
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -- ML framework --
import sklearn.model_selection
import sklearn.metrics
import sklearn.preprocessing
import sklearn.pipeline

# -- ANN framework --
import tensorflow as tf

# 1. Feed-forward neural networks: MLP and Wide & Deep.
For this initial task I will be working with the wellknown california housing dataset.

In [2]:
# Importing the housing dataset
housing = tf.keras.datasets.california_housing.load_data()
(x_train, y_train), (x_test, y_test) = housing

In [3]:
# Getting to know the data
print(
    "-- Train data info: --",
    x_train.dtype,
    x_train.shape,
    x_train[:5],
    "\n",
    "-- Test data info: --",
    x_test.dtype,
    x_test.shape,
    x_test[:5],
    sep= "\n"
)

-- Train data info: --
float32
(16512, 8)
[[-1.18270e+02  3.40900e+01  5.20000e+01  2.32700e+03  5.55000e+02
   1.04800e+03  4.91000e+02  3.78470e+00]
 [-1.18360e+02  3.39600e+01  2.10000e+01  1.80200e+03  5.56000e+02
   1.28600e+03  5.57000e+02  2.72840e+00]
 [-1.22390e+02  3.77600e+01  5.20000e+01  1.87700e+03  4.27000e+02
   7.12000e+02  3.98000e+02  3.97220e+00]
 [-1.17950e+02  3.39200e+01  1.10000e+01  3.12700e+03  7.06000e+02
   1.59400e+03  6.94000e+02  4.34260e+00]
 [-1.22520e+02  3.79200e+01  2.40000e+01  4.21000e+02  6.40000e+01
   1.63000e+02  7.50000e+01  1.45833e+01]]


-- Test data info: --
float32
(4128, 8)
[[-1.1836e+02  3.4080e+01  4.5000e+01  2.1950e+03  4.8300e+02  1.2650e+03
   4.5500e+02  3.3864e+00]
 [-1.2020e+02  3.4630e+01  1.4000e+01  2.6470e+03  5.1500e+02  1.4870e+03
   4.8800e+02  4.4519e+00]
 [-1.2121e+02  3.7810e+01  8.0000e+00  1.8830e+03  2.9800e+02  9.9900e+02
   3.0100e+02  5.1930e+00]
 [-1.2206e+02  3.7850e+01  1.7000e+01  7.4750e+03  1.5560e+03  2.09

In [4]:
# Subdividing the training set into training and validation via a conservative 80-20 split
# For features
x_tra = x_train[:-int(len(x_train) * 0.2), :]
x_val = x_train[-int(len(x_train) * 0.2):, :]

# For targets
y_tra = y_train[:-int(len(y_train) * 0.2)]
y_val = y_train[-int(len(y_train) * 0.2):]

From the housing base exploration we are able to evidence that the data comes already split between train and test sets. Hence, we would work with those initial partitions. A second property that we are able to infer is that all of the features are continuous variables. Therefore, we can assume that by default the preloaded data does not account for categorical features as in the original dataset.

## 1.1. Constructing a MLP
As this is a very straight forward ANN structure, I would implement such structure on the sequential class from keras. Class that provides a high-level implementation.

### Sequential API

**Unstandardised features model**

In [5]:
# Creating a base model (without standardization step) via the rigid sequential API
tf.random.set_seed(15) # For guaranteeing reproducibility
mlp_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=[8,]),
    tf.keras.layers.Dense(100, activation="relu"), # Hidden 1
    tf.keras.layers.Dense(50, activation="relu"), # Hidden 2
    tf.keras.layers.Dense(10, activation="relu"), # Hidden 3
    tf.keras.layers.Dense(1) # Output layer without activation function as it is an unconstrained regression task
])

# Compiling the model with default values and MSE loss function
mlp_model.compile(loss="mean_squared_error", optimizer="adam", metrics=["root_mean_squared_error"])

# Displaying the base structure
mlp_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 100)            │           900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 50)             │         5,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           510 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,471 (25.28 KB)

 Trainable params: 6,471 (25.28 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# Training the model with a low number of epochs for avoiding consuming unnecessary resources due to the lag of standardization.
history = mlp_model.fit(x=x_tra, y=y_tra, validation_data=(x_val, y_val), epochs=10)

Epoch 1/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 32431480832.0000 - root_mean_squared_error: 180087.4219 - val_loss: 26014877696.0000 - val_root_mean_squared_error: 161291.2812
Epoch 2/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 22468235264.0000 - root_mean_squared_error: 149894.0781 - val_loss: 19789733888.0000 - val_root_mean_squared_error: 140675.9844
Epoch 3/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 14699342848.0000 - root_mean_squared_error: 121240.8438 - val_loss: 11800218624.0000 - val_root_mean_squared_error: 108628.8125
Epoch 4/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 11214499840.0000 - root_mean_squared_error: 105898.5391 - val_loss: 10947206144.0000 - val_root_mean_squared_error: 104628.8984
Epoch 5/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 10858695680.0000 - root_mean_squared_error: 104205.0625 - val_loss: 10643504128.0000 - val_root_mean_squared_error: 103167.3594
Epoch 6/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 

In [7]:
np.max(history.history["loss"])

np.float64(32431480832.0)

From it we can observe that as the data has not been appropiately scale, the RMSE tends to show extreme high values on both training and validation sets. Therefore, now we will proceed to show this performance slightly improves via parsing external scaled features.

**Standardised features model**

In [8]:
# Re-scaling the features
scaler = sklearn.preprocessing.StandardScaler()
scaler = scaler.fit(x_tra)

# Implementing
std_x_tra = scaler.transform(x_tra)
std_x_val = scaler.transform(x_val)

print(
    std_x_tra[:5],
    std_x_val[:5],
    sep="\n\n"
)

[[ 0.6482476  -0.72322077  1.85232    -0.14237864  0.03717742 -0.33747274
  -0.02615596 -0.04231662]
 [ 0.603281   -0.784224   -0.6160778  -0.38027763  0.03953395 -0.1274346
   0.14472945 -0.6001574 ]
 [-1.4101332   0.9989319   1.85232    -0.34629208 -0.2644579  -0.63399714
  -0.26694903  0.05670369]
 [ 0.80812156 -0.8029945  -1.4123352   0.22013412  0.39301285  0.14437945
   0.49944615  0.25231498]
 [-1.4750808   1.074012   -0.37720057 -1.0060652  -1.1198769  -1.1184969
  -1.1032519   5.660514  ]]

[[ 1.297734   -0.8827664  -0.53645205  0.49790955  1.0033531   0.51238745
   0.8360386  -1.3524476 ]
 [-0.45088938 -0.46513274 -1.6512123  -0.07214179  0.11022973 -0.00300024
   0.10071351 -0.41922742]
 [-1.4501028   1.2476364  -1.1734579   0.4942844   0.7229265  -0.13537721
   0.70916915 -0.6860278 ]
 [ 0.6882132  -0.71383554 -0.2179491  -0.45459276 -0.19376214 -0.24304383
  -0.1659713  -0.23349181]
 [-1.3401886   1.1678628   1.4541913  -0.5900819  -0.5684498  -0.5616311
  -0.5569364  -0.6

In [9]:
# The previous model's weights are corrupted - we need to rebuild it
tf.random.set_seed(15) #For guaranteeing reproducibility
mlp_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=[8,]),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.Dense(50, activation="relu"),
    tf.keras.layers.Dense(10, activation="relu"),
    tf.keras.layers.Dense(1)
])

# Using a lower learning rate to prevent gradient explosion
mlp_model.compile(loss="mean_squared_error", optimizer="adam", metrics=["root_mean_squared_error"])

In [10]:
# Re-fitting the data in the standardized values
history = mlp_model.fit(x = std_x_tra, y = y_tra, validation_data=(std_x_val, y_val), epochs=10)

Epoch 1/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 53768491008.0000 - root_mean_squared_error: 231880.3438 - val_loss: 45876703232.0000 - val_root_mean_squared_error: 214188.4688
Epoch 2/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 25096323072.0000 - root_mean_squared_error: 158418.1875 - val_loss: 12663965696.0000 - val_root_mean_squared_error: 112534.2891
Epoch 3/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 10883056640.0000 - root_mean_squared_error: 104321.8906 - val_loss: 9401536512.0000 - val_root_mean_squared_error: 96961.5234
Epoch 4/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 8876569600.0000 - root_mean_squared_error: 94215.5469 - val_loss: 7833405952.0000 - val_root_mean_squared_error: 88506.5312
Epoch 5/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 7617438208.0000 - root_mean_squared_error: 87277.9375 - val_loss: 6766707200.0000 - val_root_mean_squared_error: 82260.0000
Epoch 6/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 6676905472

As demonstrated from the previous code, just by the fact of scaling the features we receive a boost in the performance; therefore, depicting an approximate 25% reduction in the training RMSE and about 26% in the validation RMSE. This metric reduction is explained by the fact that the gradient descent can converge faster under appropriate scaled features.

Now, despite that this could be a good starting point. The fact of having to scale the features in advance could be burdensome. Hence, instead of continuing with this API, we would model to the functional one that would enable to include some intermediate operations.

### Functional API

In [17]:
# Creating the architecture of a MLP with the functional API
tf.random.set_seed(15) #For guaranteeing reproducibility

# Creating the normalizing layer in advance for its usage in the ANN
normalizer = tf.keras.layers.Normalization()

input_ = tf.keras.layers.Input(shape=[8,])
normalized = normalizer(input_)
hidden_1 = tf.keras.layers.Dense(100, activation="relu")(normalized)
hidden_2 = tf.keras.layers.Dense(50, activation="relu")(hidden_1)
hidden_3 = tf.keras.layers.Dense(10, activation="relu")(hidden_2)
output_ = tf.keras.layers.Dense(1)(hidden_3)

# Materialising the model
mlp_model = tf.keras.Model(inputs = [input_], outputs = [output_])

# Compiling the model
mlp_model.compile(loss="mean_squared_error", optimizer="adam", metrics=["root_mean_squared_error"])

# Standardising the input
normalizer.adapt(x_tra)

# Displaying the model's architecture summary
mlp_model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ normalization_2 (Normalization) │ (None, 8)              │            17 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 100)            │           900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 50)             │         5,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 10)             │           510 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,488 (25.35 KB)

 Trainable params: 6,471 (25.28 KB)

 Non-trainable params: 17 (72.00 B)

Now that we have a ANN architecture that contemplates the standardisation of the features, we can proceed to train the models for longer and to use the callback functionalities for early stopping and for keeping the best weights and biases.

In [20]:
# Establishing the callback functionalities
checkpoints_log = tf.keras.callbacks.ModelCheckpoint("mlp_checkpoints.weights.h5", save_weights_only=True)
early_stopping_mlp = tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)

# Fitting the model
history = mlp_model.fit(x=x_tra, y=y_tra, validation_data=(x_val, y_val), epochs= 100, callbacks=[checkpoints_log, early_stopping_mlp])

Epoch 1/100
413/413 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 54316883968.0000 - root_mean_squared_error: 233059.8281 - val_loss: 48448094208.0000 - val_root_mean_squared_error: 220109.2812
Epoch 2/100
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 28308310016.0000 - root_mean_squared_error: 168250.7344 - val_loss: 13861429248.0000 - val_root_mean_squared_error: 117734.5703
Epoch 3/100
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 11569671168.0000 - root_mean_squared_error: 107562.4062 - val_loss: 9907942400.0000 - val_root_mean_squared_error: 99538.6484
Epoch 4/100
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 9294291968.0000 - root_mean_squared_error: 96406.9062 - val_loss: 8174841856.0000 - val_root_mean_squared_error: 90414.8359
Epoch 5/100
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 7907330048.0000 - root_mean_squared_error: 88923.1719 - val_loss: 7012255744.0000 - val_root_mean_squared_error: 83739.2109
Epoch 6/100
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 6900